# Unified convergence plot

This notebook reads the previously calculated k-point and sampling-time convergence statistics, then plots a unified 2 x 3 figure with consistent font, size, line, marker, error-bar, tick, and layout settings.

In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MaxNLocator


# =========================================================
# Locate project paths
# =========================================================
def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "Step_2_Uncertainty_for_independent_run").exists() and (candidate / "Step_3_time_convergence").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


ROOT = find_project_root()
K_DIR = ROOT / "Step_2_Uncertainty_for_independent_run" / "k_sensitivity_repeat_stats"
TIME_DIR = ROOT / "Step_3_time_convergence" / "sampling_time_convergence_stats"

K_ALL_CSV = K_DIR / "all_27_combinations_gamma_delta_results.csv"
K_STATS_CSV = K_DIR / "gamma_delta_statistics_over_27_combinations.csv"

TIME_ALL_CSV = TIME_DIR / "all_combinations_gamma_varepsilon_vs_sampling_time.csv"
TIME_STATS_CSV = TIME_DIR / "gamma_varepsilon_statistics_vs_sampling_time.csv"

SAVE_PNG = TIME_DIR / "unified_k_points_and_sampling_time_convergence.png"
SAVE_SVG = TIME_DIR / "unified_k_points_and_sampling_time_convergence.svg"
SAVE_PDF = TIME_DIR / "unified_k_points_and_sampling_time_convergence.pdf"


# =========================================================
# Read previous calculation results
# =========================================================
df_k_all = pd.read_csv(K_ALL_CSV).sort_values(["combo_id", "N_Points"]).copy()
df_k_stats = pd.read_csv(K_STATS_CSV).sort_values("N_Points").copy()

df_time_all = pd.read_csv(TIME_ALL_CSV).sort_values(["combo_id", "time_ns"]).copy()
df_time_stats = pd.read_csv(TIME_STATS_CSV).sort_values("time_ns").copy()


# =========================================================
# Unit conversion
# ---------------------------------------------------------
# Keep the same gamma scaling used in the previous notebooks.
# The plotted gamma_0 value is gamma_0 * 1e20.
# =========================================================
GAMMA_SCALE = 1e20

df_k_all["gamma_plot"] = df_k_all["gamma_0"] * GAMMA_SCALE
df_k_stats["gamma_mean_plot"] = df_k_stats["gamma_0_mean"] * GAMMA_SCALE
df_k_stats["gamma_std_plot"] = df_k_stats["gamma_0_std"] * GAMMA_SCALE

df_time_all["gamma_plot"] = df_time_all["gamma"] * GAMMA_SCALE
df_time_stats["gamma_mean_plot"] = df_time_stats["gamma_mean"] * GAMMA_SCALE
df_time_stats["gamma_std_plot"] = df_time_stats["gamma_std"] * GAMMA_SCALE


# =========================================================
# Unified plotting style
# =========================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 15,
    "axes.labelsize": 16,
    "axes.titlesize": 17,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "savefig.dpi": 400,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

FIGSIZE = (15.5, 9.0)

RAW_ALPHA_LINE = 0.20
RAW_ALPHA_SCATTER = 0.25
RAW_LINEWIDTH = 1.0
RAW_MARKERSIZE = 14

MEAN_LINEWIDTH = 2.6
MEAN_MARKERSIZE = 5.5
ERRORBAR_CAPSIZE = 3

MEAN_COLOR = "black"
ERR_COLOR = "black"
ERR_ALPHA = 0.60
GRID_ALPHA = 0.30


def plot_raw_and_stats(ax, raw_df, stats_df, x_col, raw_y_col, mean_y_col, std_y_col, group_col="combo_id"):
    for _, grp in raw_df.groupby(group_col):
        grp = grp.sort_values(x_col)
        ax.plot(
            grp[x_col], grp[raw_y_col],
            "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE,
        )
        ax.scatter(
            grp[x_col], grp[raw_y_col],
            s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER,
        )

    ax.errorbar(
        stats_df[x_col], stats_df[mean_y_col], yerr=stats_df[std_y_col],
        fmt="none",
        ecolor=ERR_COLOR,
        elinewidth=1.6,
        alpha=ERR_ALPHA,
        capsize=ERRORBAR_CAPSIZE,
        capthick=1.4,
        zorder=3,
    )
    ax.plot(
        stats_df[x_col], stats_df[mean_y_col],
        "-o",
        color=MEAN_COLOR,
        linewidth=MEAN_LINEWIDTH,
        markersize=MEAN_MARKERSIZE,
        zorder=4,
    )

    ax.grid(True, alpha=GRID_ALPHA)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=False))
    ax.tick_params(which="both", width=1.2)
    ax.tick_params(which="major", length=6)
    ax.tick_params(which="minor", length=3)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)


# =========================================================
# Create unified 2 x 3 figure
# =========================================================
fig, axes = plt.subplots(2, 3, figsize=FIGSIZE, constrained_layout=True)

panels = [
    {
        "ax": axes[0, 0],
        "raw_df": df_k_all,
        "stats_df": df_k_stats,
        "x_col": "N_Points",
        "raw_y_col": "gamma_plot",
        "mean_y_col": "gamma_mean_plot",
        "std_y_col": "gamma_std_plot",
        "xlabel": "Number of k points used",
        "ylabel": r"$\gamma_0$ (mJ m$^{-2}$)",
        "title": r"$\gamma_0$",
        "tag": "(a)",
    },
    {
        "ax": axes[0, 1],
        "raw_df": df_k_all,
        "stats_df": df_k_stats,
        "x_col": "N_Points",
        "raw_y_col": "delta_1",
        "mean_y_col": "delta_1_mean",
        "std_y_col": "delta_1_std",
        "xlabel": "Number of k points used",
        "ylabel": r"$\varepsilon_1$",
        "title": r"$\varepsilon_1$",
        "tag": "(b)",
    },
    {
        "ax": axes[0, 2],
        "raw_df": df_k_all,
        "stats_df": df_k_stats,
        "x_col": "N_Points",
        "raw_y_col": "delta_2",
        "mean_y_col": "delta_2_mean",
        "std_y_col": "delta_2_std",
        "xlabel": "Number of k points used",
        "ylabel": r"$\varepsilon_2$",
        "title": r"$\varepsilon_2$",
        "tag": "(c)",
    },
    {
        "ax": axes[1, 0],
        "raw_df": df_time_all,
        "stats_df": df_time_stats,
        "x_col": "time_ns",
        "raw_y_col": "gamma_plot",
        "mean_y_col": "gamma_mean_plot",
        "std_y_col": "gamma_std_plot",
        "xlabel": "Sampling time (ns)",
        "ylabel": r"$\gamma_0$ (mJ m$^{-2}$)",
        "title": r"$\gamma_0$",
        "tag": "(d)",
    },
    {
        "ax": axes[1, 1],
        "raw_df": df_time_all,
        "stats_df": df_time_stats,
        "x_col": "time_ns",
        "raw_y_col": "varepsilon_1",
        "mean_y_col": "varepsilon_1_mean",
        "std_y_col": "varepsilon_1_std",
        "xlabel": "Sampling time (ns)",
        "ylabel": r"$\varepsilon_1$",
        "title": r"$\varepsilon_1$",
        "tag": "(e)",
    },
    {
        "ax": axes[1, 2],
        "raw_df": df_time_all,
        "stats_df": df_time_stats,
        "x_col": "time_ns",
        "raw_y_col": "varepsilon_2",
        "mean_y_col": "varepsilon_2_mean",
        "std_y_col": "varepsilon_2_std",
        "xlabel": "Sampling time (ns)",
        "ylabel": r"$\varepsilon_2$",
        "title": r"$\varepsilon_2$",
        "tag": "(f)",
    },
]

for panel in panels:
    ax = panel["ax"]
    plot_raw_and_stats(
        ax=ax,
        raw_df=panel["raw_df"],
        stats_df=panel["stats_df"],
        x_col=panel["x_col"],
        raw_y_col=panel["raw_y_col"],
        mean_y_col=panel["mean_y_col"],
        std_y_col=panel["std_y_col"],
    )
    ax.set_xlabel(panel["xlabel"])
    ax.set_ylabel(panel["ylabel"])
    ax.set_title(panel["title"])
    ax.text(
        0.03, 0.94, panel["tag"],
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=16,
        fontweight="bold",
    )

fig.savefig(SAVE_PNG, bbox_inches="tight")
fig.savefig(SAVE_SVG, bbox_inches="tight")
fig.savefig(SAVE_PDF, bbox_inches="tight")

print(f"Saved figure to:\n  {SAVE_PNG}\n  {SAVE_SVG}\n  {SAVE_PDF}")
plt.show()